In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# RQ4: UMLS candidate margin (MedMentions / CADEC)

**Signal:** \(M_{\mathrm{UMLS}} = s_1 - s_2\) = cosine of the assigned CUI minus the
best cosine among retrieved forms of a *different* CUI.

This ranks inside the H=0 block that entropy cannot. No LLM re-run: stored
`output_text` is re-embedded through the PART2 SapBERT + FAISS index.

**Pooling:** PART2 index and stored `confidence` are **mean-pooled** SapBERT
(L2-normalised). Queries use the same pooling so \(s_2\) lives in the same
space as \(s_1\). (CLS would break that.)

**CUI-level, not form-level:** several surface forms share a CUI; \(s_2\) is the
max cosine over forms whose mapped CUI \(\neq\) `predicted_cui`.

Set `DATASET` in the setup cell (`medmentions` or `cadec`). CADEC mapping is
all SapBERT/FAISS (no encoder CUI head), so encoders can have a defined margin.


In [ ]:
# Setup: PART2 FAISS cache + SapBERT (same objects the mapping cell uses)
from pathlib import Path
from collections import defaultdict, Counter
import json
import gc
import re

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
import faiss

PROJECT_ROOT = PROJECT_ROOT
assert (PROJECT_ROOT / "config" / "config.json").is_file()
assert torch.cuda.is_available(), "CUDA required — refuse CPU for SapBERT"

UNASSIGNED = "UNASSIGNED"
SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
FAISS_TOP_K = int(CFG_YAML["umls"]["faiss_top_k"])  # 1000 locked
MIN_FORM_LEN = 3
_pool_config_name = f"sapbert_full_len{MIN_FORM_LEN}"
EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / _pool_config_name

import os
DATASET = os.environ.get("RQ4_MARGIN_DATASET") or os.environ.get("DATASET") or "cadec"
DATASET = str(DATASET).strip().lower()
print(f"RQ4_MARGIN_DATASET={DATASET!r} (set env to cadec|medmentions; run both)")
if DATASET == "medmentions":
    MAPPED_PATH = PROJECT_ROOT / "outputs/rq1/intermediate/rq1_all_outputs_mapped.csv"
    ENTROPY_PATH = PROJECT_ROOT / "outputs/rq1/entropy_full_umls.csv"
    OUT_PATH = PROJECT_ROOT / "outputs/rq1/umls_candidate_margin_medmentions.csv"
    # PRIMARY is the de-duplicated denominator (ANALYSIS_PRECOMMIT.md section 3). The raw
    # column stays as the fallback so an un-regenerated entropy file still loads.
    H_COL_CANDIDATES = ["normalised_entropy_dedup", "normalised_semantic_entropy_full", "normalised_semantic_entropy"]
    RETAINED_COL = "retained_m_distinct"
elif DATASET == "cadec":
    MAPPED_PATH = PROJECT_ROOT / "outputs/rq3/intermediate/rq3_cadec_mapped_outputs.csv"
    ENTROPY_PATH = PROJECT_ROOT / "outputs/rq3/entropy_cadec.csv"
    OUT_PATH = PROJECT_ROOT / "outputs/rq3/umls_candidate_margin_cadec.csv"
    H_COL_CANDIDATES = ["normalised_entropy_dedup", "normalised_entropy", "normalised_semantic_entropy"]
    RETAINED_COL = "retained_m_distinct"
else:
    raise ValueError(f"DATASET must be medmentions|cadec, got {DATASET!r}")
print(f"DATASET={DATASET}\n  mapped={MAPPED_PATH}\n  entropy={ENTROPY_PATH}\n  out={OUT_PATH}")

_forms_json = EMB_DIR / "surface_forms.json"
_pairs_json = EMB_DIR / "cui_form_pairs.json"
_index_path = EMB_DIR / "faiss.index"
for p in (_forms_json, _pairs_json, _index_path, MAPPED_PATH, ENTROPY_PATH):
    assert p.is_file(), f"missing {p}"

print(f"Loading FAISS/form maps from {EMB_DIR}")
with open(_forms_json, "r", encoding="utf-8") as f:
    _unique_forms = json.load(f)
with open(_pairs_json, "r", encoding="utf-8") as f:
    _form_cui_pairs = [tuple(x) for x in json.load(f)]
_faiss_index = faiss.read_index(str(_index_path))
assert len(_unique_forms) == _faiss_index.ntotal, (
    f"forms={len(_unique_forms)} ntotal={_faiss_index.ntotal}"
)

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_c)

def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s

print(f"FAISS {type(_faiss_index).__name__} ntotal={_faiss_index.ntotal:,}  forms={len(_unique_forms):,}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 1) Load mapped rows; keep real retrieval only
df_mapped = pd.read_csv(MAPPED_PATH, low_memory=False)
df_mapped["predicted_cui"] = df_mapped["predicted_cui"].map(_norm_cui)
df_mapped["confidence"] = pd.to_numeric(df_mapped["confidence"], errors="coerce")
if "is_direct_cui" not in df_mapped.columns:
    # CADEC: every row is SapBERT/FAISS five-rule (no encoder CUI head).
    print("NOTE: is_direct_cui column absent — treating all rows as retrieval (CADEC-style mapping)")
    df_mapped["is_direct_cui"] = False
else:
    df_mapped["is_direct_cui"] = df_mapped["is_direct_cui"].astype(bool)

print("is_direct_cui by model (1.0 = encoder CUI head, margin undefined):")
print(df_mapped.groupby("model_name")["is_direct_cui"].mean().round(3).to_string())

direct_models = (
    df_mapped.groupby("model_name")["is_direct_cui"].mean()
)
direct_models = direct_models[direct_models >= 0.99].index.tolist()
print(f"\nModels skipped (no runner-up; all/almost-all direct CUI): {direct_models}")

keep = (
    (df_mapped["is_direct_cui"] == False)
    & df_mapped["output_text"].notna()
    & (df_mapped["output_text"].astype(str).str.strip() != "")
    & (df_mapped["predicted_cui"] != UNASSIGNED)
)
df_ret = df_mapped.loc[keep].copy()
print(f"\nRetrieval rows kept: {len(df_ret):,} / {len(df_mapped):,}")
print(df_ret.groupby("model_name").size().to_string())
assert len(df_ret) > 0, "no retrieval rows — cannot compute margin"


In [ ]:
# 2–3) Re-embed output_text (mean-pool = PART2) + FAISS top-50; CUI-level s2
def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def _embed_with_model(model, tokenizer, texts, batch_size=128, max_len=64, desc="sapbert"):
    """L2-normalised mean-pooled embeddings — identical to PART2 mapping."""
    vecs = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, unit="batch"):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = tokenizer(batch, return_tensors="pt", truncation=True,
                            max_length=max_len, padding=True)
            enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
    return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)


# Local snapshot under ~/data/hf_cache (same as run_topk_sweep / PART2)
_SAP_SRC = (
    Path.home() / "data/hf_cache/hub"
    / "models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext"
    / "snapshots" / "090663c3ae57bf35ffe4d0d468a2a88d03051a4d"
)
assert (_SAP_SRC / "model.safetensors").exists() or (_SAP_SRC / "pytorch_model.bin").exists(), _SAP_SRC
print(f"Loading SapBERT from {_SAP_SRC}")
_sap_tok = AutoTokenizer.from_pretrained(str(_SAP_SRC))
_sap_mdl = AutoModel.from_pretrained(str(_SAP_SRC))
_sap_mdl = _sap_mdl.to("cuda").eval().half()
assert next(_sap_mdl.parameters()).device.type == "cuda"

# Unique texts: do not re-embed duplicates
uniq_texts = df_ret["output_text"].astype(str).unique().tolist()
print(f"Unique output_text to embed: {len(uniq_texts):,}  (from {len(df_ret):,} rows)")
uniq_vecs = _embed_with_model(_sap_mdl, _sap_tok, uniq_texts, batch_size=128, max_len=64)

print(f"FAISS search top-k={FAISS_TOP_K} on {len(uniq_vecs):,} unique texts "
      f"(map back to {len(df_ret):,} rows)")
D_u, I_u = _faiss_index.search(uniq_vecs.astype(np.float32), FAISS_TOP_K)
_text_to_ui = {t: i for i, t in enumerate(uniq_texts)}
_row_u = np.fromiter(
    (_text_to_ui[str(t)] for t in df_ret["output_text"]),
    dtype=np.int64, count=len(df_ret),
)
D, I = D_u[_row_u], I_u[_row_u]

s2_vals = np.zeros(len(df_ret), dtype=np.float32)
n_no_diff = 0
preds = df_ret["predicted_cui"].map(_norm_cui).tolist()
for i, pred in enumerate(tqdm(preds, desc="s2 cui-level")):
    best = None
    for sim, idx in zip(D[i], I[i]):
        if idx < 0:
            continue
        form = _unique_forms[int(idx)]
        cuis = {_norm_cui(c) for c in _form_to_cuis.get(form, ())}
        # s2 = nearest DIFFERENT concept. A form is admitted only if its CUI set excludes the
        # predicted CUI entirely. The previous `any(c != pred ...)` admitted polysemous forms
        # that also denote pred, inflating s2 and compressing umls_margin = s1 - s2 on exactly
        # the ambiguous mentions the margin is meant to characterise.
        if cuis and pred not in cuis and any(c != UNASSIGNED for c in cuis):
            sc = float(sim)
            if best is None or sc > best:
                best = sc
    if best is None:
        s2_vals[i] = 0.0  # no different CUI in top-k = fully unambiguous
        n_no_diff += 1
    else:
        s2_vals[i] = best

df_ret = df_ret.copy()
df_ret["s1"] = df_ret["confidence"].astype(float)
df_ret["s2"] = s2_vals
df_ret["umls_margin"] = df_ret["s1"] - df_ret["s2"]

print(f"rows with no different CUI in top-{FAISS_TOP_K} (s2=0): {n_no_diff:,} ({n_no_diff/len(df_ret):.1%})")
print(df_ret[["s1", "s2", "umls_margin"]].describe().round(4).to_string())

del _sap_mdl, uniq_vecs, D, I, D_u, I_u
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 4–6) Aggregate, merge onto entropy, sanity check inside H=0
# Pre-specified primary = margin_mean (instance-level, like entropy)
orig = (
    df_ret[df_ret["input_type"] == "original"]
    .drop_duplicates(["instance_id", "model_name"], keep="first")
    [["instance_id", "model_name", "umls_margin"]]
    .rename(columns={"umls_margin": "margin_original"})
)
df_margin = (
    df_ret.groupby(["instance_id", "model_name"], as_index=False)
    .agg(
        n_retrieval_rows=("umls_margin", "size"),
        margin_mean=("umls_margin", "mean"),
        margin_min=("umls_margin", "min"),
        s1_mean=("s1", "mean"),
        s2_mean=("s2", "mean"),
    )
    .merge(orig, on=["instance_id", "model_name"], how="left")
)

print("Per-model mean margins (retrieval models only):")
print(
    df_margin.groupby("model_name")[["margin_original", "margin_mean", "margin_min", "s1_mean", "s2_mean"]]
    .mean()
    .round(4)
    .to_string()
)

ent = pd.read_csv(ENTROPY_PATH)
# PRIMARY denominator is distinct-m (ANALYSIS_PRECOMMIT.md section 3). Filter to the retained
# row set BEFORE the merge: this CSV embeds a copy of the entropy column that
# RQ4_margin_benchmark later reads, so an unfiltered frame would propagate the raw-m row set
# downstream under a dedup column name.
if RETAINED_COL in ent.columns:
    _n0, _i0 = len(ent), ent["instance_id"].nunique()
    ent = ent[ent[RETAINED_COL].astype(bool)].copy()
    print(f"{RETAINED_COL} filter | rows {_n0:,} -> {len(ent):,} | "
          f"instances {_i0:,} -> {ent['instance_id'].nunique():,}")
else:
    print(f"WARNING: {RETAINED_COL} absent from {ENTROPY_PATH.name} — NOT filtered; "
          f"the denominator is the raw-m set")
merged = ent.merge(df_margin, on=["instance_id", "model_name"], how="left")
n_with = merged["margin_mean"].notna().sum()
n_without = merged["margin_mean"].isna().sum()
print(f"\nMerged entropy rows: {len(merged):,}  with margin={n_with:,}  without={n_without:,}")
print("Direct-CUI-skipped / without-margin counts by model:")
_missing = merged[merged["margin_mean"].isna()].groupby("model_name").size()
print(_missing.to_string() if len(_missing) else "  (none — every model×instance in entropy has a defined margin)")
print("Coverage (n with margin / n entropy) by model:")
print(
    merged.groupby("model_name")["margin_mean"]
    .agg(n="size", n_margin="count")
    .assign(frac=lambda d: (d.n_margin / d.n).round(3))
    .to_string()
)

H_COL = next((c for c in H_COL_CANDIDATES if c in merged.columns), None)
assert H_COL is not None, f"no entropy column in {list(merged.columns)}"
H = merged[H_COL].astype(float).clip(lower=0)
zero = merged[(H == 0) & merged["margin_mean"].notna()]
print(f"\nSanity — margin inside H=0 block (H col = {H_COL}):")
print(f"  n(H=0 and margin defined) = {len(zero):,}")
if len(zero) > 1:
    for col in ["margin_original", "margin_mean", "margin_min"]:
        v = zero[col].dropna()
        print(f"  {col:18s}  mean={v.mean():.4f}  std={v.std(ddof=1):.4f}  "
              f"min={v.min():.4f}  max={v.max():.4f}")
    std_mean = float(zero["margin_mean"].std(ddof=1))
    assert std_mean > 1e-4, (
        f"FAIL: margin_mean std inside H=0 is {std_mean:.6f} — margin does not see inside the zero block"
    )
    print(f"  PASS: H=0 margin_mean std = {std_mean:.4f} (non-trivial)")
    print("  H=0 margin_mean spread by model:")
    print(
        zero.groupby("model_name")["margin_mean"]
        .agg(n="count", mean="mean", std="std", min="min", max="max")
        .round(4)
        .to_string()
    )
else:
    print("  WARN: too few H=0 rows with margin")

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_PATH, index=False)
print(f"\nWrote {OUT_PATH} ({len(merged):,} rows)")
print("columns added: n_retrieval_rows, margin_original, margin_mean, margin_min, s1_mean, s2_mean")
print("primary abstention score for RQ4: margin_mean")
